In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, mne
from mne.time_frequency import psd_array_welch
from src.load import list_subjects, load_subject
from src.channels import CHANNELS, SFREQ

N = 4 * SFREQ                         # samples per 4 s segment = 512
segments, subject, label = [], [], []
for _, k in pd.DataFrame(list_subjects()).iterrows():
    x = mne.filter.filter_data(load_subject(k.path), SFREQ, 1.0, 40.0, verbose=False)
    x = x[:, :60 * SFREQ]                               # keep the first 60 s only
    segments.append(x.reshape(19, 15, N).transpose(1, 0, 2))   # -> (15 segments, 19 ch, 512)
    subject += [k.subject_id] * 15
    label += [k.label] * 15

W = np.concatenate(segments)          # all segments: (1815, 19, 512)
subject, label = np.array(subject), np.array(label)
print(W.shape)

(1815, 19, 512)


In [2]:
import torch
from torch import nn
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from src.eegnet import EEGNet

X = torch.tensor(W / 100, dtype=torch.float32).unsqueeze(1)   # scale µV to ~±1, add a "1" axis
Y = torch.tensor(label)

def run_fold(train, test, epochs=30, seed=0):
    torch.manual_seed(seed)
    net = EEGNet()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        net.train()
        order = np.random.default_rng(seed + ep).permutation(train)
        for i in range(0, len(order), 64):                 # mini-batches of 64 segments
            b = order[i:i + 64]
            opt.zero_grad()
            loss_fn(net(X[b]), Y[b]).backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        return torch.softmax(net(X[test]), dim=1)[:, 1].numpy()   # P(ADHD) per test segment

def evaluate(cv):
    p = np.zeros(len(Y))
    for f, (train, test) in enumerate(cv.split(W, label, groups=subject)):
        p[test] = run_fold(train, test)
        print("  fold", f + 1, "done")
    subject_agg = pd.DataFrame({"subject": subject, "y": label, "p": p}).groupby("subject").mean()
    return roc_auc_score(label, p), roc_auc_score(subject_agg.y, subject_agg.p)

for name, cv in [("segment-wise (leaky)", StratifiedKFold(5, shuffle=True, random_state=42)),
                 ("subject-wise", StratifiedGroupKFold(5, shuffle=True, random_state=42))]:
    segment_auc, subject_auc = evaluate(cv)
    print(f"{name:7s} segment AUC {segment_auc:.2f}   per-subject AUC {subject_auc:.2f}")

C:\Users\Rajesh Sampathkumar\miniconda3\envs\medimg\Lib\site-packages\sklearn\model_selection\_split.py:884: UserWarning: The groups parameter is ignored by StratifiedKFold
  warnings.warn(


  fold 1 done


  fold 2 done


  fold 3 done


  fold 4 done


...

  fold 1 done


  fold 2 done


  fold 3 done


  fold 4 done


  fold 5 done
subject-wise segment AUC 0.73   per-subject AUC 0.77
